# 20. Os números do artigo

Junta num lugar só os números que o artigo cita, todos da base diária real
(`data/mercado_diario.db`) e calculados com os módulos já prontos. Nada aqui
passa para o app/. As estatísticas dos dados, os testes e a figura deles saem
do statsmodels, e cada número é conferido contra a conta feita à mão.
**F2, F6, F11, NF1, NF4.**

In [1]:
import sys, os, time
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np, pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
from statsmodels.stats.descriptivestats import Description
from statsmodels.stats.weightstats import DescrStatsW
from statsmodels.tsa.stattools import acf
from statsmodels.graphics.tsaplots import plot_acf
from app import dal, nucleo
from app.agente import Investidor
from app.mercado import RendaVariavel
from app.principal import executar_pipeline

DB = os.path.join(RAIZ, 'data', 'mercado_diario.db')
PPA = 252

## 1. A amostra

**Teste**: a base diária tem que ter 1.050 pregões, do primeiro retorno, em
24/05/2022, até 05/08/2026.

In [2]:
ret = dal.ler_sqlite(DB, 'retornos')
n = len(ret)
print('pregoes:', n, '| de', ret['data'].iloc[0], 'a', ret['data'].iloc[-1])

pregoes: 1050 | de 2022-05-24 a 2026-08-05


In [3]:
assert n == 1050 and ret['data'].iloc[0] == '2022-05-24' and ret['data'].iloc[-1] == '2026-08-05'

## 2. Estatísticas descritivas

Com o `Description` do statsmodels, que dá numa tabela só a média, o
erro-padrão dela, o desvio, a mediana, a assimetria, a curtose e o Jarque-Bera.
A média vai para o ano composta, (1 + m)^252 - 1, e o desvio vai multiplicado
pela raiz de 252, porque a variância da soma de 252 retornos independentes é
252 vezes a de um. O statsmodels dá a curtose inteira; o excesso é ela menos 3,
que é a da normal.

A razão t é a média dividida pelo erro-padrão, e o p-valor dela sai do teste t do
`DescrStatsW`: se a média verdadeira fosse zero, a razão seguiria uma t de
Student com n - 1 graus de liberdade, e o p-valor é a chance de cair tão longe de
zero quanto caiu. Como os retornos não são normais, é um p-valor aproximado.

O Jarque-Bera junta assimetria (g1) e excesso de curtose (g2) num número só,
JB = n/6 * (g1^2 + g2^2/4), que fica perto de zero numa normal.

In [4]:
series = {'Ibovespa': ret['ibov'].to_numpy(), 'CDI': ret['cdi'].to_numpy(),
          'excesso': (ret['ibov'] - ret['cdi']).to_numpy()}

d = Description(pd.DataFrame(series),
                stats=['nobs', 'mean', 'std_err', 'std', 'min', 'median', 'max',
                       'skew', 'kurtosis', 'jarque_bera']).frame
descritivas = pd.DataFrame({
    'n': d.loc['nobs'],
    'media': d.loc['mean'], 'media_aa': (1 + d.loc['mean']) ** PPA - 1,
    'erro_padrao': d.loc['std_err'], 'razao_t': d.loc['mean'] / d.loc['std_err'],
    'p_t': pd.Series({k: DescrStatsW(v).ttest_mean(0)[1] for k, v in series.items()}),
    'desvio': d.loc['std'], 'desvio_aa': d.loc['std'] * np.sqrt(PPA),
    'minimo': d.loc['min'], 'mediana': d.loc['median'], 'maximo': d.loc['max'],
    'assimetria': d.loc['skew'], 'curtose_excesso': d.loc['kurtosis'] - 3,
    'jarque_bera': d.loc['jarque_bera'], 'jb_p': d.loc['jarque_bera_pval']}).T.astype(float)
print(descritivas.to_string(float_format=lambda v: f'{v:.6g}'))

                   Ibovespa         CDI     excesso
n                      1050        1050        1050
media           0.000511037 0.000489236 2.18008e-05
media_aa           0.137404    0.131176  0.00550885
erro_padrao     0.000329822 1.63994e-06 0.000329771
razao_t             1.54943     298.325   0.0661088
p_t                0.121579           0    0.947304
desvio            0.0106874 5.31402e-05   0.0106858
desvio_aa          0.169658 0.000843575    0.169632
minimo           -0.0430936   0.0003927  -0.0436449
mediana         0.000295442  0.00050788 -0.00018325
maximo            0.0554086  0.00055131   0.0549008
assimetria        0.0777007   -0.602891   0.0746641
curtose_excesso      1.3345   -0.927703      1.3363
jarque_bera         78.9707     101.261     79.0993
jb_p            7.10765e-18 1.02656e-22 6.66497e-18


**Teste**: o statsmodels tem que dar o mesmo número que as fórmulas do texto,
feitas aqui à mão. O Jarque-Bera
também, pela fórmula de cima, e o p-valor do t, pela t de Student.

In [5]:
def descreve(x):
    return {'media': x.mean(), 'erro_padrao': x.std(ddof=1) / np.sqrt(len(x)),
            'desvio': x.std(ddof=1), 'minimo': x.min(), 'mediana': np.median(x),
            'maximo': x.max(), 'assimetria': stats.skew(x),
            'curtose_excesso': stats.kurtosis(x)}

na_mao = pd.DataFrame({k: descreve(v) for k, v in series.items()})
g1, g2 = descritivas.loc['assimetria'], descritivas.loc['curtose_excesso']
jb_na_mao = n / 6 * (g1 ** 2 + g2 ** 2 / 4)
print(na_mao.to_string(float_format=lambda v: f'{v:.6g}'))
p_t_na_mao = 2 * stats.t.sf(np.abs(descritivas.loc['razao_t']), n - 1)
print('JB pela formula:', jb_na_mao.round(2).to_dict())
print('p-valor do t pela t de Student:', {k: round(float(v), 4) for k, v in zip(series, p_t_na_mao)})

                   Ibovespa         CDI     excesso
media           0.000511037 0.000489236 2.18008e-05
erro_padrao     0.000329822 1.63994e-06 0.000329771
desvio            0.0106874 5.31402e-05   0.0106858
minimo           -0.0430936   0.0003927  -0.0436449
mediana         0.000295442  0.00050788 -0.00018325
maximo            0.0554086  0.00055131   0.0549008
assimetria        0.0777007   -0.602891   0.0746641
curtose_excesso      1.3345   -0.927703      1.3363
JB pela formula: {'Ibovespa': 78.97, 'CDI': 101.26, 'excesso': 79.1}
p-valor do t pela t de Student: {'Ibovespa': 0.1216, 'CDI': 0.0, 'excesso': 0.9473}


In [6]:
assert np.allclose(na_mao, descritivas.loc[na_mao.index], rtol=1e-9, atol=0)
assert np.allclose(jb_na_mao, descritivas.loc['jarque_bera'])
assert np.allclose(p_t_na_mao, descritivas.loc['p_t'])

O CDI e o Ibovespa em cada ano civil: a taxa média do CDI levada ao ano e o
retorno acumulado dos dois dentro do ano (2022 começa em maio e 2026 termina
em agosto).

In [7]:
por_ano = ret.assign(ano=ret['data'].str[:4]).groupby('ano').agg(
    pregoes=('cdi', 'size'),
    cdi_medio_aa=('cdi', lambda x: (1 + x.mean()) ** PPA - 1),
    cdi_acumulado=('cdi', lambda x: np.prod(1 + x) - 1),
    ibov_acumulado=('ibov', lambda x: np.prod(1 + x) - 1))
print(por_ano.to_string(float_format=lambda v: f'{v:.4f}'))

      pregoes  cdi_medio_aa  cdi_acumulado  ibov_acumulado
ano                                                       
2022      153        0.1343         0.0795         -0.0029
2023      248        0.1321         0.1299          0.2195
2024      251        0.1082         0.1078         -0.1036
2025      250        0.1432         0.1420          0.3395
2026      148        0.1456         0.0831          0.1030


## 3. Autocorrelação

Dos retornos e dos retornos ao quadrado, nas defasagens de 1 a 10, com o `acf`
do statsmodels. Se os retornos fossem independentes, cada autocorrelação seria
mais ou menos a média de n produtos sem correlação entre si, com variância perto
de 1/n, e quase todas ficariam dentro da faixa de 2 / raiz(n).

O `qstat=True` dá junto o Ljung-Box, Q(k) = n(n+2) * soma_{j<=k} rho_j^2 / (n-j),
que testa se as k primeiras autocorrelações são todas zero.

In [8]:
r_ibov = series['Ibovespa']
ac_r, q_r, p_r = acf(r_ibov, nlags=10, qstat=True)
ac_q, q_q, p_q = acf(r_ibov ** 2, nlags=10, qstat=True)
ac = pd.DataFrame({'retorno': ac_r[1:], 'Q_retorno': q_r, 'p_retorno': p_r,
                   'quadrado': ac_q[1:], 'Q_quadrado': q_q, 'p_quadrado': p_q},
                  index=range(1, 11))
faixa_ac = 2 / np.sqrt(n)
print(ac.round(4).to_string())
print(f'faixa de 2/raiz(n): {faixa_ac:.4f}')

    retorno  Q_retorno  p_retorno  quadrado  Q_quadrado  p_quadrado
1   -0.0005     0.0003     0.9861    0.0553      3.2146      0.0730
2   -0.0247     0.6447     0.7244    0.0030      3.2242      0.1995
3    0.0103     0.7559     0.8600    0.0502      5.8856      0.1173
4    0.0580     4.3036     0.3665    0.0117      6.0298      0.1969
5   -0.0517     7.1336     0.2109    0.0624     10.1485      0.0711
6    0.0054     7.1643     0.3059    0.0891     18.5540      0.0050
7    0.0147     7.3944     0.3890    0.0656     23.1184      0.0016
8    0.0257     8.0953     0.4242    0.0141     23.3287      0.0030
9    0.0159     8.3646     0.4979    0.0647     27.7696      0.0010
10   0.0001     8.3646     0.5933    0.0772     34.0971      0.0002
faixa de 2/raiz(n): 0.0617


**Teste**: o `acf` tem que bater com a correlação da série com ela mesma
defasada. As duas só diferem no denominador (o
statsmodels usa a amostra inteira), então a folga é de 0.005. E o Q tem que sair
igual pela fórmula de cima.

In [9]:
def autocorr(x, k):
    return np.corrcoef(x[:-k], x[k:])[0, 1]

ac_na_mao = np.array([[autocorr(r_ibov, k), autocorr(r_ibov ** 2, k)] for k in range(1, 11)])
j = np.arange(1, 11)
q_na_mao = n * (n + 2) * np.cumsum(ac_q[1:] ** 2 / (n - j))
print('maior diferenca no acf:', np.abs(ac_na_mao - ac[['retorno', 'quadrado']].to_numpy()).max().round(5))

maior diferenca no acf: 0.00018


In [10]:
assert np.allclose(ac_na_mao, ac[['retorno', 'quadrado']], atol=0.005)
assert np.allclose(q_na_mao, q_q)

## 4. O prêmio do Ibovespa sobre o CDI

O erro-padrão da média é o desvio dividido pela raiz de n, porque a variância
da média de n observações independentes é sigma^2 / n. A razão t é a média
dividida por esse erro. Aqui com o `DescrStatsW` do statsmodels.

In [11]:
ds = DescrStatsW(series['excesso'])
premio, ep_premio = ds.mean, ds.std_mean
t_premio = premio / ep_premio
premio_aa = descritivas.loc['media_aa', 'Ibovespa'] - descritivas.loc['media_aa', 'CDI']
print(f'premio por pregao: {premio:.7f} | erro-padrao: {ep_premio:.7f} | t = {t_premio:.2f}')
print(f'diferenca das medias ao ano: {100 * premio_aa:.2f} p.p.')

premio por pregao: 0.0000218 | erro-padrao: 0.0003298 | t = 0.07
diferenca das medias ao ano: 0.62 p.p.


**Teste**: o erro-padrão do `DescrStatsW` tem que ser o desvio sobre a raiz de
n, e tem que bater com o da tabela do item 2.

In [12]:
assert np.isclose(ep_premio, series['excesso'].std(ddof=1) / np.sqrt(n))
assert np.isclose(ep_premio, descritivas.loc['erro_padrao', 'excesso'])

## 5. A rodada principal

É a mesma do `python -m app --diario`, só que com 40 milhões de cenários:
gamma = 5, beta = 0.96 ao ano, 5 anos (T = 1.260 pregões), W_0 = 1, 3.000
trajetórias e semente 1. Rodo três vezes para medir o tempo, e as três têm que
dar o mesmo número (NF4).

In [13]:
GAMMA, BETA_ANUAL, ANOS, SEMENTE = 5.0, 0.96, 5, 1
N_PRINCIPAL = 40_000_000
CFG = {'db_path': DB, 'tabela': 'retornos', 'ativos': ['ibov'], 'periodos_por_ano': PPA,
       'gamma': GAMMA, 'beta_anual': BETA_ANUAL, 'w0': 1.0, 'horizonte': PPA * ANOS,
       'n_scenarios': N_PRINCIPAL, 'n_paths': 3_000, 'seed': SEMENTE}

tempos_principal, alphas_repetidos = [], []
for _ in range(3):
    t0 = time.perf_counter()
    res = executar_pipeline(CFG)
    tempos_principal.append(time.perf_counter() - t0)
    alphas_repetidos.append(res['alpha_star'][0])

rf, mu = res['rf'], res['mu_hat'][0]
sigma2 = res['sigma_hat'][0, 0]
alpha_principal = res['alpha_star'][0]
alpha_merton = (mu - rf) / (GAMMA * sigma2)
print(f'alpha* = {alpha_principal:.4f} | Merton (mu - R_f)/(gamma sigma^2) = {alpha_merton:.4f}')
print(f"Phi = {res['phi_hat']:.6f} | beta por pregao = {res['beta']:.6f}")
print(f"theta_0 = {res['theta'][0]:.6f} | theta_T = {res['theta'][-1]:.4f}")
print('consumo por ano (fracao de W_0):', res['consumo_por_ano'].round(4))
print(f"E[W_T] = {res['E_W_T']:.6f} [P5 = {res['W_T_p5']:.6f}, P95 = {res['W_T_p95']:.6f}]")

alpha* = 0.0430 | Merton (mu - R_f)/(gamma sigma^2) = 0.0382
Phi = 0.998043 | beta por pregao = 0.999838
theta_0 = 0.001024 | theta_T = 1.0000
consumo por ano (fracao de W_0): [0.2601 0.2645 0.269  0.2735 0.2781]
E[W_T] = 0.001113 [P5 = 0.001084, P95 = 0.001144]


In [14]:
assert len(set(alphas_repetidos)) == 1

## 6. Quanto o alpha* muda com a semente

Resolvo só a carteira (Etapa 1) com 100 sementes e 4 milhões de cenários, que é
o padrão da base diária, e com 100 sementes e 40 milhões, que é o da rodada
principal. Nos dois casos as sementes vão de 1 em diante, então a semente 1 com 40
milhões é a própria rodada principal.

De cada caso tiro a média, o desvio, o mínimo e o máximo do alpha* e o tempo
médio de uma resolução. O erro-padrão da média é o desvio sobre a raiz do número
de sementes, pelo mesmo argumento do item 4. A célula dos 40 milhões é a
mais demorada do notebook.

In [15]:
mercado = RendaVariavel(ret[['data', 'ibov']])
inv = Investidor(GAMMA, res['beta'], 1.0, PPA * ANOS)
N_SEMENTES_4M, N_SEMENTES_40M = 100, 100

def alpha_por_semente(n_cen, sementes):
    alphas, tempos = [], []
    for s in sementes:
        t0 = time.perf_counter()
        alphas.append(inv.carteira_otima(mercado, rf, n_scenarios=n_cen, seed=s)[0])
        tempos.append(time.perf_counter() - t0)
    return np.array(alphas), np.array(tempos)

alphas_4m, tempos_4m = alpha_por_semente(4_000_000, range(1, N_SEMENTES_4M + 1))
print(f'4 milhoes, {N_SEMENTES_4M} sementes: media {alphas_4m.mean():.4f}, desvio {alphas_4m.std(ddof=1):.4f}')

4 milhoes, 100 sementes: media 0.0393, desvio 0.0097


In [16]:
alphas_40m, tempos_40m = alpha_por_semente(N_PRINCIPAL, range(1, N_SEMENTES_40M + 1))
print(f'40 milhoes, {N_SEMENTES_40M} sementes: media {alphas_40m.mean():.4f}, desvio {alphas_40m.std(ddof=1):.4f}')

40 milhoes, 100 sementes: media 0.0384, desvio 0.0030


In [17]:
def resume(alphas, tempos):
    k = len(alphas)
    return {'sementes': k, 'media': alphas.mean(),
            'erro_padrao': alphas.std(ddof=1) / np.sqrt(k), 'desvio': alphas.std(ddof=1),
            'minimo': alphas.min(), 'maximo': alphas.max(), 'tempo_s': tempos.mean()}

sementes = pd.DataFrame({'4 milhoes': resume(alphas_4m, tempos_4m),
                         '40 milhoes': resume(alphas_40m, tempos_40m)})
dp_4m, dp_40m = sementes.loc['desvio']
razao = dp_40m / dp_4m
print(sementes.to_string(float_format=lambda v: f'{v:.4f}'))
print(f'razao dos desvios: {razao:.3f} | 1/raiz(10) = {1 / np.sqrt(10):.3f}')

             4 milhoes  40 milhoes
sementes      100.0000    100.0000
media           0.0393      0.0384
erro_padrao     0.0010      0.0003
desvio          0.0097      0.0030
minimo          0.0152      0.0314
maximo          0.0617      0.0455
tempo_s         1.4849     13.9755
razao dos desvios: 0.309 | 1/raiz(10) = 0.316


**Teste**: com 10 vezes mais cenários o desvio entre sementes tem que cair com
1 sobre a raiz de n, então a razão fica perto de 1/raiz(10), aprox 0.316. Só que
o próprio desvio vem com erro, de uns 1/raiz(2(k-1)): 7% com 100 sementes, nos dois
casos. Juntando os dois, a razão erra uns 10%, e com folga de três vezes isso
ela tem que ficar entre 0.22 e 0.41.

E a semente 1 com 40 milhões tem que dar o mesmo alpha* da rodada principal.

In [18]:
assert 0.22 < razao < 0.41
assert np.isclose(alphas_40m[0], alpha_principal, rtol=0, atol=1e-12)

**Teste**: o alpha* da rodada principal e o da fórmula de Merton têm que ficar a
menos de três desvios (os de 40 milhões) um do outro. A fórmula é de tempo
contínuo, e num pregão ela deveria quase coincidir com a solução discreta.

In [19]:
print(f'distancia em desvios: {abs(alpha_principal - alpha_merton) / dp_40m:.2f}')
assert abs(alpha_principal - alpha_merton) < 3 * dp_40m

distancia em desvios: 1.62


**Teste**: a média dos 100 alpha* com 40 milhões tira o ruído de uma semente só
e tem erro-padrão de uns 0.0003. Ela tem que ficar a menos de três erros-padrão
da fórmula de Merton. Na aproximação de primeira ordem que o artigo usa, as duas
soluções só diferem pelo fator R_f, que num pregão é 1.0005, então a diferença
esperada é de uns 0.00002, bem menor que esse erro.

In [20]:
media_40m, ep_media_40m = sementes.loc['media', '40 milhoes'], sementes.loc['erro_padrao', '40 milhoes']
dist_media = abs(media_40m - alpha_merton) / ep_media_40m
print(f'media com 40 milhoes: {media_40m:.4f} (erro-padrao {ep_media_40m:.4f}) | '
      f'Merton: {alpha_merton:.4f} | distancia em erros-padrao: {dist_media:.2f}')

media com 40 milhoes: 0.0384 (erro-padrao 0.0003) | Merton: 0.0382 | distancia em erros-padrao: 0.72


In [21]:
assert dist_media < 3

Agora os alpha* de cada semente num gráfico só: uma linha para cada número de
cenários, com um ponto por semente, a fórmula de Merton tracejada e a rodada
principal em destaque. Vai para `results/alpha_sementes.png`.

In [22]:
fig, ax = plt.subplots(figsize=(7, 3.2))
for y, alphas in [(0, alphas_4m), (1, alphas_40m)]:
    ax.scatter(alphas, np.full(len(alphas), y), s=40,
               color='#1f77b4', alpha=0.25, lw=0)
ax.axvline(alpha_merton, color='0.45', ls='--', lw=1)
ax.annotate(f'regra de Merton ({alpha_merton:.4f})', (alpha_merton, -0.48), xytext=(5, 0),
            textcoords='offset points', ha='left', va='center', fontsize=8, color='0.25')
ax.scatter([alpha_principal], [1], s=60, marker='D', color='#d62728', zorder=5)
ax.annotate(f'rodada principal ({alpha_principal:.4f})', (alpha_principal, 1), xytext=(10, 14),
            textcoords='offset points', fontsize=8, color='0.25')
ax.set_yticks([0, 1], ['4 milhões\nde cenários', '40 milhões\nde cenários'])
ax.set_ylim(-0.65, 1.55)
ax.set_xlabel(r'$\alpha^*$')
ax.set_title(r'Carteira ótima $\alpha^*$ em cada semente')
for lado in ('top', 'right'):
    ax.spines[lado].set_visible(False)
fig.tight_layout()
fig.text(0.5, 0.0, f"Ibovespa diário (dados reais) · {ret['data'].iloc[0]} a {ret['data'].iloc[-1]} "
         f"({n} obs) · $\\gamma$={GAMMA:g} · {N_SEMENTES_4M} sementes com 4 milhões e "
         f"{N_SEMENTES_40M} com 40 milhões de cenários",
         ha='center', va='top', fontsize=7, color='0.45')
FIG_SEMENTES = os.path.join(RAIZ, 'results', 'alpha_sementes.png')
fig.savefig(FIG_SEMENTES, dpi=150, bbox_inches='tight')
plt.close(fig)
print('gravada em', os.path.relpath(FIG_SEMENTES, RAIZ))

gravada em article\figs\alpha_sementes.png


**Teste**: a figura tem que existir e ter um ponto para cada semente.

In [23]:
assert os.path.getsize(FIG_SEMENTES) > 0
assert len(alphas_4m) == N_SEMENTES_4M and len(alphas_40m) == N_SEMENTES_40M

## 7. O alpha* com a incerteza do prêmio

O mu_chapeu vem de uma amostra, e o erro-padrão dele (item 4) é bem maior que o
erro de Monte Carlo. Para ver o tamanho disso, desloco os cenários da rodada
principal para prêmios de 2 erros-padrão abaixo e acima do estimado e resolvo de
novo, com a mesma amostra. Pela fórmula de Merton, o erro do prêmio passa para
o alpha* dividido por gamma * sigma^2.

In [24]:
r = mercado.amostrar(N_PRINCIPAL, seed=SEMENTE)
Rf = 1.0 + rf
faixa = {k: nucleo.resolver_alpha_otimo(np.maximum(1.0 + r + k * ep_premio, 0.0), Rf, GAMMA)[0]
         for k in (-2, 0, 2)}
del r
ep_alpha_merton = ep_premio / (GAMMA * sigma2)
print('alpha* com premio - 2 ep, estimado, + 2 ep:', {k: round(v, 4) for k, v in faixa.items()})
print(f'erro-padrao do alpha* pela formula de Merton: {ep_alpha_merton:.3f}')

alpha* com premio - 2 ep, estimado, + 2 ep: {-2: np.float64(-1.1124), 0: np.float64(0.043), 2: np.float64(1.1984)}
erro-padrao do alpha* pela formula de Merton: 0.577


## 8. Etapa 7 na rodada principal

**Teste**: as duas conferências do projeto, agora com os dados reais. O A_t da
recorrência tem que bater com a forma fechada, e a utilidade que a política
simulada entrega tem que bater com o V_0(W_0), com a mesma identidade de
Bellman do nb11:

V_0(W_0) = sum_{s<t} beta^s E[u(c_s)] + beta^t E[V_t(W_t)], para todo t.

In [25]:
T, beta = res['horizonte'], res['beta']
bphi = beta * res['phi_hat']
t = np.arange(T + 1)
A_fechada = ((1 - bphi ** ((T - t + 1) / GAMMA)) / (1 - bphi ** (1 / GAMMA))) ** GAMMA
desvio_A = np.max(np.abs(res['A_t'] / A_fechada - 1))

def desvio_bellman(res):
    b, T = res['beta'], res['horizonte']
    u_med, V_med = res['trajetoria_u_media'], res['trajetoria_V_media']
    desc = b ** np.arange(T + 1)
    acumulado = np.concatenate([[0.0], np.cumsum(desc[:-1] * u_med[:-1])])
    return np.max(np.abs((acumulado + desc * V_med) / V_med[0] - 1))

desvio_V = desvio_bellman(res)
print(f'A_t: maior desvio relativo contra a forma fechada = {desvio_A:.2e}')
print(f"V_0(W_0) = {res['trajetoria_V_media'][0]:.6e} | maior desvio relativo na identidade = {desvio_V:.2e}")

A_t: maior desvio relativo contra a forma fechada = 1.01e-12
V_0(W_0) = -2.223987e+14 | maior desvio relativo na identidade = 1.13e-03


In [26]:
assert desvio_A < 1e-10 and desvio_V < 0.02

## 9. Tempos

A esteira inteira (a rodada principal, três vezes) e só a Etapa 1, com 4 e com
40 milhões de cenários. Os tempos dependem da máquina em que o notebook rodou.

In [27]:
print(f'esteira inteira, 40 milhoes: {np.mean(tempos_principal):.1f} s em media')
print(f'so a Etapa 1: {tempos_4m.mean():.1f} s com 4 milhoes, {tempos_40m.mean():.1f} s com 40 milhoes')

esteira inteira, 40 milhoes: 14.7 s em media
so a Etapa 1: 1.5 s com 4 milhoes, 14.0 s com 40 milhoes


## 10. A figura dos dados

Quatro painéis: (a) a série dos retornos diários, (b) o gráfico quantil-quantil
contra a normal, (c) e (d) as autocorrelações dos retornos e dos quadrados, com
a faixa de 2 / raiz(n). Os painéis b, c e d são do statsmodels. Vai para
`results/`, junto com as outras figuras.

In [28]:
datas = pd.to_datetime(ret['data'])
fig, eixos = plt.subplots(2, 2, figsize=(10, 7))

ax = eixos[0, 0]
ax.plot(datas, 100 * r_ibov, lw=0.6, color='#1f77b4')
ax.axhline(0, color='0.6', lw=0.6)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_title('(a) Retornos diários do Ibovespa')
ax.set_ylabel('retorno (%)')

ax = eixos[0, 1]
sm.qqplot(r_ibov, dist=stats.norm, fit=True, line='45', ax=ax, markersize=3, alpha=0.6)
ax.set_xlim(-3.6, 3.6); ax.set_ylim(-4.6, 5.6)
ax.set_title('(b) Quantis da amostra contra os da normal')
ax.set_xlabel('quantis da normal padrão')
ax.set_ylabel('quantis da amostra padronizada')

for ax, x, titulo in [(eixos[1, 0], r_ibov, '(c) Autocorrelação dos retornos'),
                      (eixos[1, 1], r_ibov ** 2, '(d) Autocorrelação dos retornos ao quadrado')]:
    plot_acf(x, lags=10, alpha=None, zero=False, ax=ax, title=titulo)
    for s in (1, -1):
        ax.axhline(s * faixa_ac, color='#d62728', ls='--', lw=0.8)
    ax.set_xticks(range(1, 11)); ax.set_ylim(-0.12, 0.12)
    ax.set_xlabel('defasagem (pregões)')

fig.tight_layout()
fig.text(0.5, 0.0, f"Ibovespa diário (dados reais) · {ret['data'].iloc[0]} a {ret['data'].iloc[-1]} "
         f"({n} obs) · tracejado em $\\pm 2/\\sqrt{{n}}$ = $\\pm${faixa_ac:.3f}",
         ha='center', va='top', fontsize=7, color='0.45')
FIG = os.path.join(RAIZ, 'results', 'diagnostico_retornos.png')
fig.savefig(FIG, dpi=150, bbox_inches='tight')
plt.close(fig)
print('gravada em', os.path.relpath(FIG, RAIZ))

gravada em article\figs\diagnostico_retornos.png


**Teste**: a figura tem que existir, e os pontos dos painéis c e d têm que ser
os mesmos números do item 3.

In [29]:
assert os.path.getsize(FIG) > 0
assert np.allclose(ac_r[1:], ac['retorno']) and np.allclose(ac_q[1:], ac['quadrado'])

## Resumo

Os números que o artigo cita saem todos daqui.

In [30]:
def aa(x):
    return (1 + x) ** PPA - 1
print(f"amostra: {n} pregoes, {ret['data'].iloc[0]} a {ret['data'].iloc[-1]}")
print(f'Ibovespa: media {100 * mu:.4f}% por pregao ({100 * aa(mu):.2f}% a.a.), '
      f'desvio {100 * np.sqrt(sigma2):.3f}% ({100 * np.sqrt(sigma2 * PPA):.2f}% a.a.)')
print(f'CDI (R_f): {100 * rf:.4f}% por pregao ({100 * aa(rf):.2f}% a.a.)')
print(f'premio: {premio:.7f} por pregao, erro-padrao {ep_premio:.7f}, t = {t_premio:.2f}; '
      f'{100 * premio_aa:.2f} p.p. a.a.')
print()
print('tabela dos dados no artigo (em % quando for retorno):')
pct = ['media', 'erro_padrao', 'desvio', 'minimo', 'mediana', 'maximo']
tab = descritivas.copy()
tab.loc[pct] *= 100
tab.loc[['media_aa', 'desvio_aa']] *= 100
print(tab.to_string(float_format=lambda v: f'{v:.4f}'))
print(f'Ljung-Box Q(10): retornos {q_r[-1]:.2f} (p = {p_r[-1]:.3f}) | '
      f'quadrados {q_q[-1]:.2f} (p = {p_q[-1]:.4f})')
print(f'autocorrelacoes: retornos de {ac_r[1:].min():.3f} a {ac_r[1:].max():.3f}; '
      f'quadrados de {ac_q[1:].min():.3f} a {ac_q[1:].max():.3f}, '
      f'{int((ac_q[1:] > faixa_ac).sum())} acima da faixa de {faixa_ac:.3f}')
print()
print(f'alpha* principal {alpha_principal:.4f} | Merton {alpha_merton:.4f}')
print('tabela das sementes no artigo:')
print(sementes.to_string(float_format=lambda v: f'{v:.4f}'))
print(f'razao dos desvios {razao:.2f} | principal a {abs(alpha_principal - alpha_merton) / dp_40m:.2f} '
      f'desvios de Merton | media com 40 mi a {dist_media:.2f} erros-padrao de Merton')
print(f"alpha* com premio -2 ep e +2 ep: {faixa[-2]:.3f} e {faixa[2]:.3f}")
print(f"Phi {res['phi_hat']:.6f} | beta por pregao {beta:.6f} | theta_0 {res['theta'][0]:.6f}")
print('consumo por ano:', res['consumo_por_ano'].round(4))
print(f'desvio A_t {desvio_A:.1e} | desvio Bellman {desvio_V:.1e}')
print(f'tempo da esteira {np.mean(tempos_principal):.1f} s')

amostra: 1050 pregoes, 2022-05-24 a 2026-08-05
Ibovespa: media 0.0511% por pregao (13.74% a.a.), desvio 1.069% (16.97% a.a.)
CDI (R_f): 0.0489% por pregao (13.12% a.a.)
premio: 0.0000218 por pregao, erro-padrao 0.0003298, t = 0.07; 0.62 p.p. a.a.

tabela dos dados no artigo (em % quando for retorno):
                 Ibovespa       CDI   excesso
n               1050.0000 1050.0000 1050.0000
media              0.0511    0.0489    0.0022
media_aa          13.7404   13.1176    0.5509
erro_padrao        0.0330    0.0002    0.0330
razao_t            1.5494  298.3251    0.0661
p_t                0.1216    0.0000    0.9473
desvio             1.0687    0.0053    1.0686
desvio_aa         16.9658    0.0844   16.9632
minimo            -4.3094    0.0393   -4.3645
mediana            0.0295    0.0508   -0.0183
maximo             5.5409    0.0551    5.4901
assimetria         0.0777   -0.6029    0.0747
curtose_excesso    1.3345   -0.9277    1.3363
jarque_bera       78.9707  101.2613   79.0993
jb_p    